In [ ]:
# Install dependencies (Run once)
# !pip install -q openai gradio

from openai import OpenAI
import gradio as gr

# Import your files
from context import MED_SYSTEM_PROMPT
from styles import CSS, JS, EXAMPLES

# -------------------------------
# Configuration
# -------------------------------

OLLAMA_BASE_URL = "http://localhost:11434/v1"

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"  # Dummy key (ignored by Ollama)
)

MODEL_NAME = "medgemma:latest"

system_message = [
    {
        "role": "system",
        "content": MED_SYSTEM_PROMPT
    }
]

# -------------------------------
# Chat Function
# -------------------------------

def chat(message, history):
    """
    message : Latest user message
    history : Previous chat history from Gradio
    """

    messages = system_message.copy()

    # Convert Gradio history into OpenAI format
    for user_msg, assistant_msg in history:
        messages.append(
            {
                "role": "user",
                "content": user_msg
            }
        )
        messages.append(
            {
                "role": "assistant",
                "content": assistant_msg
            }
        )

    messages.append(
        {
            "role": "user",
            "content": message
        }
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

    return response.choices[0].message.content

# -------------------------------
# Launch Gradio
# -------------------------------

demo = gr.ChatInterface(
    fn=chat,
    examples=EXAMPLES,
    title="🩺 MedAssist",
    description="AI Healthcare Assistant powered by MedGemma running locally via Ollama.",
    chatbot=gr.Chatbot(show_label=False),
)

demo.launch(
    css=CSS,
    js=JS,
    theme=gr.themes.Base(),
    share=False,
    inbrowser=True
)